# Project 2 — Notebook 01: Data Cleaning & EDA

**Infotact Data Analytics Internship — Cohort Retention & CLTV Analysis**

**Week 1 goal:** turn three separate dataset files into ONE clean, trustworthy table
we can use for cohort analysis in Week 2.

This notebook (started on **Day 2**) covers:

1. Loading all three dataset parts
2. Understanding why the files have different columns (raw vs. cleaned)
3. Combining them into one table with a single, consistent set of columns
4. Removing the overlap where the raw files and the cleaned file meet
5. Beginning the cleaning process (duplicates, cancellations, invalid rows, missing IDs)
6. Recording exactly how many rows each step removed (an "audit trail")

> **Reminder:** the data files live in a local `data/` folder that is **never uploaded to GitHub**
> (blocked by `.gitignore`). Only this notebook and its written outputs are committed.


## Step 1 — Import the tools

`pandas` is the main tool for working with tables of data. `numpy` helps with numbers and missing values. We give them short nicknames (`pd`, `np`) because everyone in data analysis writes them that way.

In [ ]:
import pandas as pd
import numpy as np

# Show a few more rows/columns when we preview tables
pd.set_option('display.max_columns', 30)
pd.set_option('display.width', 140)

print('pandas version:', pd.__version__)

## Step 2 — Load the three dataset parts

We read every column as text (`dtype=str`) on purpose. If we let pandas guess the types
too early, it can silently mangle IDs and dates. We will convert to proper numbers and
dates *deliberately* in the cleaning steps.

- `dataset_part_1.csv` and `dataset_part_2.csv` — **raw** data, 8 columns
- `dataset_part_3_cleaned.csv` — a **pre-cleaned** slice with 13 columns

In [ ]:
DATA_DIR = 'data'  # local folder (git-ignored)

d1 = pd.read_csv(f'{DATA_DIR}/dataset_part_1.csv', dtype=str)
d2 = pd.read_csv(f'{DATA_DIR}/dataset_part_2.csv', dtype=str)
d3 = pd.read_csv(f'{DATA_DIR}/dataset_part_3_cleaned.csv', dtype=str)

print('part 1 shape:', d1.shape)
print('part 2 shape:', d2.shape)
print('part 3 shape:', d3.shape)
print()
print('RAW columns (part 1/2):', list(d1.columns))
print()
print('CLEANED columns (part 3):', list(d3.columns))

## Step 3 — Why do the files look different?

The three files are **time-slices of the same dataset** (a UK online gift retailer):

| File | Type | Columns | Covers |
|---|---|---|---|
| part 1 + part 2 | raw | 8 | Dec 2010 → 26 Sep 2011 |
| part 3 | pre-cleaned | 13 | 26 Sep 2011 → 09 Dec 2011 |

The raw files use capitalised names like `InvoiceNo`, `CustomerID`.
The cleaned file uses lower-case names like `invoice_no`, `customer_id`, and adds extra
helper columns (`line_total`, `invoice_status`, `line_type`, `data_quality_flags`).

To combine them we must **rename the raw columns to match** and keep only the columns
that exist in *both*. This shared set of columns is called our **canonical schema**
("schema" just means the agreed list of column names).

## Step 4 — Put the raw files into the canonical schema

We rename raw columns, convert numbers/dates, and add a `line_total` column
(`quantity × unit_price`) so the raw data matches the cleaned file's structure.
We also add a `source_file` label so we can always tell where a row came from.

In [ ]:
def standardize_raw(df):
    """Rename raw 8-column data into our canonical lower-case schema."""
    out = pd.DataFrame({
        'invoice_no':       df['InvoiceNo'].astype(str).str.strip(),
        'stock_code':       df['StockCode'].astype(str).str.strip(),
        'description':      df['Description'],
        'quantity':         pd.to_numeric(df['Quantity'], errors='coerce'),
        'invoice_datetime': pd.to_datetime(df['InvoiceDate'], errors='coerce'),
        'unit_price':       pd.to_numeric(df['UnitPrice'], errors='coerce'),
        'customer_id':      df['CustomerID'].astype(str).str.strip(),
        'country':          df['Country'],
    })
    out['line_total'] = (out['quantity'] * out['unit_price']).round(2)
    out['source_file'] = 'raw_parts_1_2'
    return out

raw = pd.concat([standardize_raw(d1), standardize_raw(d2)], ignore_index=True)
print('combined raw rows:', len(raw))
raw.head(3)

## Step 5 — Put the cleaned file into the same schema

Part 3 already uses the right column names, so we just select the shared columns,
convert types, and add the same `source_file` label.

In [ ]:
part3 = pd.DataFrame({
    'invoice_no':       d3['invoice_no'].astype(str).str.strip(),
    'stock_code':       d3['stock_code'].astype(str).str.strip(),
    'description':      d3['description'],
    'quantity':         pd.to_numeric(d3['quantity'], errors='coerce'),
    'invoice_datetime': pd.to_datetime(d3['invoice_datetime'], errors='coerce'),
    'unit_price':       pd.to_numeric(d3['unit_price'], errors='coerce'),
    'customer_id':      d3['customer_id'].astype(str).str.strip(),
    'country':          d3['country'],
    'line_total':       pd.to_numeric(d3['line_total'], errors='coerce'),
})
part3['source_file'] = 'cleaned_part_3'
print('cleaned part 3 rows:', len(part3))
part3.head(3)

## Step 6 — ⚠️ Fix the overlap where the files meet (important!)

**Problem discovered on Day 2:** the raw files *end* on the exact same timestamp that the
cleaned file *begins* — `2011-09-26 15:28:00`. The final invoice in the raw data,
**invoice `568346`**, also appears at the very start of the cleaned file.

If we simply glued the files together, that invoice's items would be **counted twice** —
which would quietly corrupt every later calculation.

**The fix (a clean "seam"):** keep raw rows only for dates **strictly before** the moment
part 3 starts, then attach part 3 from that moment onward. Each timestamp is then covered
by exactly one source.

In [ ]:
# The moment the cleaned file starts
p3_start = part3['invoice_datetime'].min()
print('Cleaned part 3 starts at:', p3_start)

# Keep raw rows strictly BEFORE that moment (drops the duplicated seam invoice)
raw_keep = raw[raw['invoice_datetime'] < p3_start].copy()
print('raw rows total :', len(raw))
print('raw rows kept  :', len(raw_keep), '  (dropped', len(raw) - len(raw_keep), 'overlapping seam rows)')

## Step 7 — Combine into one continuous table

Now we stack the kept raw rows on top of the cleaned rows. We also convert the text `'nan'` / empty strings in `customer_id` and `description` into real 'missing' markers so pandas recognises them as missing.

In [ ]:
combined = pd.concat([raw_keep, part3], ignore_index=True)

# Turn leftover 'nan'/'' text into real missing values
combined['customer_id'] = combined['customer_id'].replace({'nan': np.nan, '': np.nan})
combined['description'] = combined['description'].replace({'nan': np.nan})

print('COMBINED rows:', f"{len(combined):,}")
print('date range   :', combined['invoice_datetime'].min(), '->', combined['invoice_datetime'].max())
print('countries    :', combined['country'].nunique())
combined.head(3)

## Step 8 — Sanity check: is the timeline continuous?

A cohort analysis needs an unbroken run of months. Let's confirm every month from Dec 2010 to Dec 2011 is present (no gaps at the seam).

In [ ]:
months = combined['invoice_datetime'].dt.to_period('M').value_counts().sort_index()
print(months)

## Step 9 — Look at the data quality problems (before cleaning)

Before removing anything, we measure the problems. This becomes evidence in our README and shows *why* each cleaning step is needed.

In [ ]:
print('Missing customer_id :', combined['customer_id'].isna().sum(),
      f"({combined['customer_id'].isna().mean()*100:.1f}%)")
print('Missing description :', combined['description'].isna().sum())
print('Cancellation invoices (start with C):', combined['invoice_no'].str.startswith('C').sum())
print('Negative-quantity rows (returns)   :', (combined['quantity'] < 0).sum())
print('Zero/negative unit_price rows      :', (combined['unit_price'] <= 0).sum())
print('Exact duplicate rows               :', combined.duplicated().sum())

## Step 10 — Clean the data (with an audit trail)

We now remove bad rows **one step at a time** and record how many rows remain after each
step. This audit trail is exactly the kind of transparent, documented process the
project grading rewards.

Order and reasoning:
1. **Duplicates** — identical repeated rows add no information.
2. **Cancellations** (`invoice_no` starts with `C`) — these are refunds/returns, not purchases.
3. **Non-positive quantity** — returns / adjustments; keep only real sales (`quantity > 0`).
4. **Non-positive price** — free items / data errors; keep only `unit_price > 0`.
5. **Missing `customer_id`** — cohort analysis tracks a *customer* over time, so a row with
   no customer cannot be used for retention. We remove these **for the cohort dataset**.

In [ ]:
audit = []

def log_step(name, df):
    audit.append({'step': name, 'rows_remaining': len(df)})

work = combined.copy()
log_step('0. combined (start)', work)

# 1. duplicates
work = work.drop_duplicates()
log_step('1. drop duplicate rows', work)

# 2. cancellations
work = work[~work['invoice_no'].str.startswith('C')]
log_step('2. remove cancellation invoices', work)

# 3. positive quantity
work = work[work['quantity'] > 0]
log_step('3. keep quantity > 0', work)

# 4. positive price
work = work[work['unit_price'] > 0]
log_step('4. keep unit_price > 0', work)

# 5. require customer id
work = work[work['customer_id'].notna()]
log_step('5. require customer_id present', work)

# recompute line_total so it is always consistent
work['line_total'] = (work['quantity'] * work['unit_price']).round(2)

audit_df = pd.DataFrame(audit)
audit_df['rows_removed'] = audit_df['rows_remaining'].shift(1) - audit_df['rows_remaining']
audit_df

## Step 11 — Final cleaned dataset summary

A quick profile of the clean table we will carry into Week 2.

In [ ]:
clean = work.copy()

print('FINAL CLEANED DATASET')
print('---------------------')
print('rows            :', f"{len(clean):,}")
print('unique customers:', f"{clean['customer_id'].nunique():,}")
print('unique invoices :', f"{clean['invoice_no'].nunique():,}")
print('date range      :', clean['invoice_datetime'].min(), '->', clean['invoice_datetime'].max())
print('countries       :', clean['country'].nunique())
print('total revenue   : £', f"{clean['line_total'].sum():,.2f}")
clean.head(3)

## Step 12 — Notes, decisions & what comes next

**Key decisions made today (Day 2):**
- The three files were combined into one canonical table (`invoice_no, stock_code,
  description, quantity, invoice_datetime, unit_price, customer_id, country, line_total`).
- A duplicated **seam invoice (`568346`, 2011-09-26 15:28:00)** was found and removed by
  keeping raw rows only *before* part 3 begins — preventing double-counting.
- Cleaning removed duplicates, cancellations, non-positive quantity/price, and
  missing-customer rows, leaving a fully continuous 13-month sales table.

**Assumption to revisit later:** rows with a missing `customer_id` are removed *for the
cohort/CLTV work* because retention must follow a known customer. They could still be used
for total-sales EDA if ever needed.

**Next (Week 1 continues):** add the `transaction_month` and each customer's
`cohort_month` (first purchase month) columns — the foundation for the Week 2 retention
matrix.

> Before committing this notebook to GitHub, use **Kernel → Restart & Clear Output** so the
> saved file tracks *code*, not big data outputs (an Infotact requirement).